In [ ]:
import pandas as pd

# ajuste o caminho e o separador conforme o seu arquivo
df = pd.read_csv(
    "arquivo_filtrado_smq_erro_de_medicacao_y.csv",
    sep=",",          
    encoding="utf-8", 
    low_memory=False
)


In [ ]:
DRUG_COL = "Harmonização"

aware_lookup_filled = pd.read_excel("Mapeamento_aware.xlsx",
    dtype={DRUG_COL: "string"}  # garante o mesmo tipo da coluna de merge
)

aware_lookup_filled[DRUG_COL] = (
    aware_lookup_filled[DRUG_COL]
    .astype("string")
    .str.strip()
)

df[DRUG_COL] = (
    df[DRUG_COL]
    .astype("string")
    .str.strip()
)


In [ ]:
n_before = df.shape[0]

df = df.merge(
    aware_lookup_filled,
    on=DRUG_COL,
    how="left",
    validate="m:1"
)

n_after = df.shape[0]
n_before, n_after


In [ ]:
df["aware_class"].value_counts(dropna=False)


In [ ]:
aware_categories = [
    "ACCESS",
    "WATCH",
    "RESERVE",
    "NOT CLASSIFIED",
    "NOT RECOMMENDED"
]

df["aware_class"] = pd.Categorical(
    df["aware_class"],
    categories=aware_categories,
    ordered=False
)


Criar o desfecho “erro de dose” no nível do par

In [ ]:
EVENT_COL = "PT"

dose_error_terms = {
    "Administração de dose não ajustada",
    "Confusão com a dose do produto",
    "Confusão quanto ao regime do produto",
    "Dosagem incorreta administrada",
    "Dosagem não ajustada",
    "Dose adicional administrada",
    "Dose aumentada administrada",
    "Dose de reforço perdida",
    "Dose incorreta",
    "Dose incorreta administrada",
    "Dose incorreta administrada pelo dispositivo",
    "Dose incorreta administrada pelo produto",
    "Dose subterapêutica acidental",
    "Duração incorreta de administração do produto",
    "Erro de cálculo da dose",
    "Erro de titulação de medicamento",
    "Intoxicação acidental",
    "Omissão de dose do medicamento pelo dispositivo",
    "Omissão de dose do produto por erro",
    "Posologia inadequada de administração de produto",
    "Regime posológico incorreto",
    "Superdosagem acidental",
    "Taxa incorreta",
    "Taxa incorreta de administração do medicamento",
    "Titulação de dose do medicamento não realizada",
    "Dose subterapêutica",
    "Dose subterapêutica de radiação",
    "Dose subterapêutica prescrita",
    "Exposição a doses radioativas excessivas",
    "Problema relacionado à omissão de dose do produto",
    "Superdosagem",
    "Superdosagem por prescrição médica"
}

df["erro_dose"] = df[EVENT_COL].isin(dose_error_terms).astype(int)
df["erro_dose"].value_counts()


In [ ]:
#Preparar dataset para análise comparativa (excluindo as 2 categorias raras)
df_model = df[df["aware_class"].isin(["ACCESS", "WATCH", "RESERVE"])].copy()
df_model["aware_class"].value_counts()


In [ ]:
n_total = df["erro_dose"].shape[0]
n_err = df["erro_dose"].sum()
p_err = n_err / n_total

n_total, n_err, p_err


In [ ]:
tab = (
    df.groupby("aware_class")["erro_dose"]
      .agg(n="size", n_erro="sum", prop_erro="mean")
      .sort_values("n", ascending=False)
)

tab
tab.assign(prop_erro_pct=tab["prop_erro"]*100)

In [ ]:
# Aplicar sua regra: análises comparativas apenas com 3 categorias principais
df_model = df[df["aware_class"].isin(["ACCESS", "WATCH", "RESERVE"])].copy()

df_model["aware_class"].value_counts()
df_model["erro_dose"].value_counts()


In [ ]:
# ================================
# Distribuição por AWaRe 
# ================================

import numpy as np

# 1) Totais por AWaRe (todos os erros)
dist_all = (
    df.groupby("aware_class", dropna=False)
      .size()
      .reset_index(name="n_all")
)

total_all = int(dist_all["n_all"].sum())
dist_all["pct_all"] = 100 * dist_all["n_all"] / total_all if total_all > 0 else 0.0

# 2) Totais por AWaRe (somente erros de dose)
dist_dose = (
    df.loc[df["erro_dose"] == 1]
      .groupby("aware_class", dropna=False)
      .size()
      .reset_index(name="n_dose")
)

total_dose = int(dist_dose["n_dose"].sum())
dist_dose["pct_dose"] = 100 * dist_dose["n_dose"] / total_dose if total_dose > 0 else 0.0

# 3) Unir tabelas (NÃO usar fillna(0) no DF inteiro por causa do Categorical)
dist_summary = dist_all.merge(dist_dose, on="aware_class", how="left")

# Preencher NA apenas nas colunas numéricas geradas pelo merge
for col in ["n_dose", "pct_dose"]:
    if col in dist_summary.columns:
        dist_summary[col] = dist_summary[col].fillna(0)

dist_summary["n_dose"] = dist_summary["n_dose"].astype(int)

# 4) Funções de formatação
fmt_pct = lambda x: f"{float(x):.1f}%"
fmt_n = lambda x: f"{int(x):,}".replace(",", ",")

# 5) Helper seguro (se alguma classe não existir, dá erro claro)
def get_row(cls):
    sub = dist_summary.loc[dist_summary["aware_class"] == cls]
    if sub.empty:
        raise ValueError(f"Classe '{cls}' não encontrada em aware_class no dataframe.")
    return sub.iloc[0]

access  = get_row("ACCESS")
watch   = get_row("WATCH")
reserve = get_row("RESERVE")

# 6) Classes residuais
unclass = dist_summary.loc[
    dist_summary["aware_class"].isin(["NOT CLASSIFIED", "NOT RECOMMENDED"])
]

n_unclass_all  = int(unclass["n_all"].sum())
n_unclass_dose = int(unclass["n_dose"].sum())
pct_unclass_all = (100 * n_unclass_all / total_all) if total_all > 0 else 0.0

# (Opcional) Mostrar tabela-resumo para conferência
dist_summary


## Modelo principal — razões de prevalência (PR) do erro de dose por classe AWaRe
Regressão de Poisson com variância clusterizada por notificação; ajustada por idade, sexo e tipo de entrada no VigiMed (referência AWaRe = Access). Inclui a análise de sensibilidade restrita a medicamentos suspeitos.

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

ID_COL   = "IDENTIFICACAO_NOTIFICACAO"
AGE_COL  = "FAIXA_ETARIA"
ENTRY_COL = "TIPO_ENTRADA_VIGIMED"  
Y_COL    = "erro_dose"
REL_COL = "RELACAO_MEDICAMENTO_EVENTO_y"

# =========================
# Função de mapping (copiada da Tabela 1)
# =========================
def apply_entry_vigimed_mapping(
    df: pd.DataFrame,
    col_entry: str,
    out_col: str = "ENTRY_EN",
):
    # 1) Limpeza profunda
    def clean_text(x):
        if pd.isna(x) or x == "":
            return ""
        x = str(x)
        x = x.replace("\xa0", " ").replace("\u00A0", " ")
        x = " ".join(x.split())
        return x

    df = df.copy()
    df[col_entry] = df[col_entry].apply(clean_text)

    # 2) Dicionário de tradução (colapsando indústria -> Pharmaceutical companies)
    entry_map = {
        "Empresas Farmacêuticas": "Pharmaceutical companies",
        "Pacientes e Profissionais de Saúde": "Patients and healthcare professionals",
        "Serviços de Saúde": "Healthcare services",
        "Serviços de Vacinação": "Vaccination services",
        "VigiFlow eForms": "VigiFlow eForms",
        "VigiMobile": "VigiMobile",
        "eReporting - Indústria, Carga E2B": "Pharmaceutical companies",
        "eReporting - Indústria, Entrada manual de dados": "Pharmaceutical companies",
    }

    # 3) Mapeamento
    df[out_col] = df[col_entry].map(entry_map)

    # 4) Não mapeado -> Unknown
    df[out_col] = df[out_col].fillna("Not reported / Unknown")

    # 5) Ordem final (estável)
    entry_order = [
        "Healthcare services",
        "Pharmaceutical companies",
        "Patients and healthcare professionals",
        "VigiMobile",
        "VigiFlow eForms",
        "Vaccination services",
        "Not reported / Unknown",
    ]
    existing_cats = set(df[out_col].unique())
    final_cats = [x for x in entry_order if x in existing_cats] + [
        x for x in existing_cats if x not in entry_order
    ]
    df[out_col] = pd.Categorical(df[out_col], categories=final_cats, ordered=True)

    return df

# 1) Subset (3 classes) e cópia
df_model = df[df["aware_class"].isin(["ACCESS", "WATCH", "RESERVE"])].copy()

# 2) ELIMINAR pd.NA do pandas (patsy/statsmodels não lida bem)
df_model = df_model.replace({pd.NA: np.nan})

# 3) Garantir que variáveis categóricas sejam "object" (não pandas StringDtype / Categorical)
df_model["aware_class"] = df_model["aware_class"].astype(object)
df_model[AGE_COL]       = df_model[AGE_COL].astype(object)
df_model[ENTRY_COL]     = df_model[ENTRY_COL].astype(object)      # <- novo

# 4) Padronização + missing como categoria explícita
df_model["aware_class"] = pd.Series(df_model["aware_class"]).str.strip().str.upper()
df_model[AGE_COL]       = pd.Series(df_model[AGE_COL]).str.strip().fillna("MISSING")
df_model[ENTRY_COL]     = pd.Series(df_model[ENTRY_COL]).astype(object)               # <- mantido simples

# 5) Desfecho e cluster (checar missing aqui também)
df_model[Y_COL] = df_model[Y_COL].fillna(0).astype(int)

# Se existir alguma notificação sem ID, remova (cluster não pode ser NA)
df_model = df_model.dropna(subset=[ID_COL])

# >>> NOVO: aplicar o mapping do entry para criar ENTRY_EN
df_model = apply_entry_vigimed_mapping(df_model, ENTRY_COL, out_col="ENTRY_EN")

# >>> EXCLUI vacinação (zero eventos / separação perfeita)
df_model = df_model[df_model["ENTRY_EN"] != "Vaccination services"].copy()
df_model["ENTRY_EN"] = df_model["ENTRY_EN"].cat.remove_unused_categories()

# --- SEXO -> SEX_EN
SEX_COL = "SEXO"
def map_sex(x):
    if pd.isna(x):
        return "Not reported / Unknown"
    x = str(x).strip().upper()
    if "FEM" in x:
        return "Female"
    if "MASC" in x:
        return "Male"
    return "Not reported / Unknown"
df_model["SEX_EN"] = df_model[SEX_COL].apply(map_sex)

#Selecionar só suspeito
df_sens = df_model[df_model[REL_COL] == "Suspeito"].copy()
df_sens["ENTRY_EN"] = df_sens["ENTRY_EN"].cat.remove_unused_categories()

# Checagem final
print(df_model[[ID_COL, "aware_class", AGE_COL, "ENTRY_EN", Y_COL]].isna().sum())
print(len(df_sens))                                   # novo N — reporte na carta
print(pd.crosstab(df_sens["aware_class"], df_sens[Y_COL]))
print(pd.crosstab(df_sens["ENTRY_EN"], df_sens[Y_COL]))   # célula zerada = decidir colapsar/remover


# 6) Rodar 
# >>> MUDOU: removeu gravidade e incluiu ENTRY_EN
formula = (
    f"{Y_COL} ~ "
    f"C(aware_class, Treatment(reference='ACCESS')) + "
    f"C({AGE_COL}, Treatment(reference='Adulto (19-64 anos)')) + "
    f"C(ENTRY_EN) + "
     f"C(SEX_EN, Treatment(reference='Male'))"
)

glm = smf.glm(
    formula=formula,
    data=df_model,
    family=sm.families.Poisson()
)

res = glm.fit(
    cov_type="cluster",
    cov_kwds={"groups": df_model[ID_COL]}
)

print(res.summary())

# EXTRAÇÃO DOS DADOS (PR)
params = res.params
conf = res.conf_int()

pr_table = pd.DataFrame({
    "term": params.index,
    "PR": np.exp(params.values),
    "CI95_low": np.exp(conf[0].values),
    "CI95_high": np.exp(conf[1].values),
    "p_value": res.pvalues.values
})

pr_table_no_intercept = pr_table[pr_table["term"] != "Intercept"].copy()
display(pr_table_no_intercept)

#modelo sensibilidade (somente suspeito)
glm_sens = smf.glm(formula=formula, data=df_sens, family=sm.families.Poisson())
res_sens = glm_sens.fit(cov_type="cluster", cov_kwds={"groups": df_sens[ID_COL]})
print(res_sens.summary())

params_s = res_sens.params
conf_s = res_sens.conf_int()
pr_table_sens = pd.DataFrame({
    "term": params_s.index,
    "PR": np.exp(params_s.values),
    "CI95_low": np.exp(conf_s[0].values),
    "CI95_high": np.exp(conf_s[1].values),
    "p_value": res_sens.pvalues.values
})
pr_table_sens_no_intercept = pr_table_sens[pr_table_sens["term"] != "Intercept"].copy()
pr_table_sens_no_intercept

In [ ]:
# ==============================================================================
# EXPORTAR TABELA
# ==============================================================================
import re
import pandas as pd
from docx import Document
from docx.shared import Pt
from docx.oxml.ns import qn

# 1) Copiar e limpar
tbl = pr_table_no_intercept.copy()

def clean_term(term: str):
    # AWaRe
    if "aware_class" in term:
        if "RESERVE" in term: return ("AWaRe class", "Reserve")
        if "WATCH" in term:   return ("AWaRe class", "Watch")

    # AGE
    if "FAIXA_ETARIA" in term:
        m = re.search(r"\[T\.(.*)\]", term)
        if m:
            label = m.group(1).strip()
            label = label.replace("Adolescente (13-18 anos)", "Adolescent (13–18)")
            label = label.replace("Criança (6-12 anos)", "Child (6–12)")
            label = label.replace("Infantil (31 dias - 5 anos)", "Infant (31 days–5 years)")
            label = label.replace("Neonato (0-30 dias)", "Neonate (0–30 days)")
            label = label.replace("Idoso (65+ anos)", "Older adult (≥65)")
            label = label.replace("Adulto (19-64 anos)", "Adult (19–64)")
            label = label.replace("Ignorado", "Not reported / Unknown")
            label = label.replace("MISSING", "Not reported / Unknown")
            return ("Age group", label)

    # ENTRY_EN
    if "ENTRY_EN" in term:
        m = re.search(r"\[T\.(.*)\]", term)
        if m:
            label = m.group(1).strip()
            if label == "Unknown":
                label = "Not reported / Unknown"
            return ("Reporting source", label)

    # SEX
    if "SEX_EN" in term:
        m = re.search(r"\[T\.(.*)\]", term)
        if m:
            return ("Sex", m.group(1).strip())

    return ("Other", term)
        
tbl[["Variable", "Category"]] = tbl["term"].apply(lambda x: pd.Series(clean_term(str(x))))

# Formatação
def fmt_pr_ci(pr_val, lo, hi, nd=2):
    return f"{pr_val:.{nd}f} ({lo:.{nd}f}–{hi:.{nd}f})"

def fmt_p(p):
    if pd.isna(p): return ""
    return "<0.001" if p < 0.001 else f"{p:.3f}"

tbl["PR (95% CI)"] = tbl.apply(lambda r: fmt_pr_ci(r["PR"], r["CI95_low"], r["CI95_high"]), axis=1)
tbl["p-value"] = tbl["p_value"].apply(fmt_p)

tbl_clean = tbl[["Variable", "Category", "PR (95% CI)", "p-value"]].copy()

# ==============================================================================
# REFERÊNCIAS (ATUALIZADAS)
# ==============================================================================
refs = pd.DataFrame([
    {"Variable": "AWaRe class",      "Category": "Access (reference)",            "PR (95% CI)": "Reference", "p-value": ""},
    {"Variable": "Age group",        "Category": "Adult (19–64; reference)",      "PR (95% CI)": "Reference", "p-value": ""},
    {"Variable": "Reporting source", "Category": "Healthcare services (reference)","PR (95% CI)": "Reference", "p-value": ""},
    {"Variable": "Sex", "Category": "Male (reference)", "PR (95% CI)": "Reference", "p-value": ""},
])

final_table = pd.concat([refs, tbl_clean], ignore_index=True)

# ==============================================================================
# ORDENAÇÃO CONTROLADA (inclui Not reported / Unknown)
# ==============================================================================
var_order = {"AWaRe class": 0, "Age group": 1, "Reporting source": 2, "Sex": 3, "Other": 9}
age_order = [
    "Neonate (0–30 days)",
    "Infant (31 days–5 years)",
    "Child (6–12)",
    "Adolescent (13–18)",
    "Adult (19–64; reference)",   # só para a linha referência (não virá dos termos)
    "Older adult (≥65)",
    "Not reported / Unknown",
]
entry_order = [
    "Healthcare services (reference)",  # referência (não virá dos termos)
    "Healthcare services",
    "Pharmaceutical companies",
    "Patients and healthcare professionals",
    "VigiMobile",
    "VigiFlow eForms",
    "Vaccination services",
    "Not reported / Unknown",
]
sex_order = ["Male (reference)", "Female", "Not reported / Unknown"]
aware_order = [
    "Access (reference)",
    "Reserve",
    "Watch",
]

def cat_rank(row):
    v = row["Variable"]
    c = row["Category"]
    if v == "AWaRe class":
        return aware_order.index(c) if c in aware_order else 99
    if v == "Age group":
        return age_order.index(c) if c in age_order else 99
    if v == "Reporting source":
        return entry_order.index(c) if c in entry_order else 99
    return 99
    if v == "Sex":
        return sex_order.index(c) if c in sex_order else 99

final_table["_vord"] = final_table["Variable"].map(var_order).fillna(9).astype(int)
final_table["_cord"] = final_table.apply(cat_rank, axis=1)

final_table = final_table.sort_values(["_vord", "_cord"], kind="stable").drop(columns=["_vord", "_cord"])

# ==============================================================================
# EXPORTAR DOCX
# ==============================================================================
out_path = "Table_y_AWaRe_main.docx"
doc = Document()

style = doc.styles["Normal"]
style.font.name = "Times New Roman"
style._element.rPr.rFonts.set(qn("w:eastAsia"), "Times New Roman")
style.font.size = Pt(11)

title = "Table. Association between AWaRe class and dose-related medication errors among antibiotic medication error reports"
p = doc.add_paragraph(title)
p.runs[0].bold = True

cols = ["Variable", "Category", "PR (95% CI)", "p-value"]
table = doc.add_table(rows=1, cols=len(cols))
table.style = "Table Grid"

hdr = table.rows[0].cells
for j, c in enumerate(cols):
    hdr[j].text = c
    hdr[j].paragraphs[0].runs[0].bold = True

last_var = None
for _, r in final_table.iterrows():
    row = table.add_row().cells
    var = r["Variable"]
    row[0].text = var if var != last_var else ""
    row[1].text = str(r["Category"])
    row[2].text = str(r["PR (95% CI)"])
    row[3].text = str(r["p-value"])
    last_var = var

doc.add_paragraph("")

note = (
    "Note: Adjusted prevalence ratios (PR) and 95% confidence intervals (CI) were estimated using "
    "Poisson regression with log link and robust variance (clustered by notification "
    "(IDENTIFICACAO_NOTIFICACAO)). Models were adjusted for age group, sex, and reporting source. "
    "Reference categories: Access (AWaRe class), Adult (19–64) (age), Male (sex), "
    "Healthcare services (reporting source)."
)
note_p = doc.add_paragraph(note)
note_p.runs[0].italic = True

doc.save(out_path)
print(f"Saved Poisson table: {out_path}")

In [ ]:
# ==============================================================================
# EXPORTAR TABELA — SENSIBILIDADE (somente medicamentos SUSPEITOS)
# Fonte: pr_table_sens_no_intercept  |  SEM GRAVIDADE + ENTRY_EN + SEXO
# ==============================================================================
import re
import pandas as pd
from docx import Document
from docx.shared import Pt
from docx.oxml.ns import qn

tbl = pr_table_sens_no_intercept.copy()

def clean_term(term: str):
    if "aware_class" in term:
        if "RESERVE" in term: return ("AWaRe class", "Reserve")
        if "WATCH" in term:   return ("AWaRe class", "Watch")
    if "FAIXA_ETARIA" in term:
        m = re.search(r"\[T\.(.*)\]", term)
        if m:
            label = m.group(1).strip()
            label = label.replace("Adolescente (13-18 anos)", "Adolescent (13–18)")
            label = label.replace("Criança (6-12 anos)", "Child (6–12)")
            label = label.replace("Infantil (31 dias - 5 anos)", "Infant (31 days–5 years)")
            label = label.replace("Neonato (0-30 dias)", "Neonate (0–30 days)")
            label = label.replace("Idoso (65+ anos)", "Older adult (≥65)")
            label = label.replace("Adulto (19-64 anos)", "Adult (19–64)")
            label = label.replace("Ignorado", "Not reported / Unknown")
            label = label.replace("MISSING", "Not reported / Unknown")
            return ("Age group", label)
    if "ENTRY_EN" in term:
        m = re.search(r"\[T\.(.*)\]", term)
        if m:
            label = m.group(1).strip()
            if label == "Unknown":
                label = "Not reported / Unknown"
            return ("Reporting source", label)
    if "SEX_EN" in term:
        m = re.search(r"\[T\.(.*)\]", term)
        if m:
            return ("Sex", m.group(1).strip())
    return ("Other", term)

tbl[["Variable", "Category"]] = tbl["term"].apply(lambda x: pd.Series(clean_term(str(x))))

def fmt_pr_ci(pr_val, lo, hi, nd=2):
    return f"{pr_val:.{nd}f} ({lo:.{nd}f}–{hi:.{nd}f})"

def fmt_p(p):
    if pd.isna(p): return ""
    return "<0.001" if p < 0.001 else f"{p:.3f}"

tbl["PR (95% CI)"] = tbl.apply(lambda r: fmt_pr_ci(r["PR"], r["CI95_low"], r["CI95_high"]), axis=1)
tbl["p-value"] = tbl["p_value"].apply(fmt_p)
tbl_clean = tbl[["Variable", "Category", "PR (95% CI)", "p-value"]].copy()

refs = pd.DataFrame([
    {"Variable": "AWaRe class",      "Category": "Access (reference)",             "PR (95% CI)": "Reference", "p-value": ""},
    {"Variable": "Age group",        "Category": "Adult (19–64; reference)",       "PR (95% CI)": "Reference", "p-value": ""},
    {"Variable": "Sex",              "Category": "Male (reference)",               "PR (95% CI)": "Reference", "p-value": ""},
    {"Variable": "Reporting source", "Category": "Healthcare services (reference)","PR (95% CI)": "Reference", "p-value": ""},
])
final_table = pd.concat([refs, tbl_clean], ignore_index=True)

var_order = {"AWaRe class": 0, "Age group": 1, "Sex": 2, "Reporting source": 3, "Other": 9}
age_order = ["Neonate (0–30 days)", "Infant (31 days–5 years)", "Child (6–12)",
             "Adolescent (13–18)", "Adult (19–64; reference)", "Older adult (≥65)",
             "Not reported / Unknown"]
sex_order = ["Male (reference)", "Female", "Not reported / Unknown"]
entry_order = ["Healthcare services (reference)", "Healthcare services", "Pharmaceutical companies",
               "Patients and healthcare professionals", "VigiMobile", "VigiFlow eForms",
               "Vaccination services", "Not reported / Unknown"]
aware_order = ["Access (reference)", "Reserve", "Watch"]

def cat_rank(row):
    v, c = row["Variable"], row["Category"]
    if v == "AWaRe class":      return aware_order.index(c) if c in aware_order else 99
    if v == "Age group":        return age_order.index(c)   if c in age_order   else 99
    if v == "Sex":              return sex_order.index(c)   if c in sex_order   else 99
    if v == "Reporting source": return entry_order.index(c) if c in entry_order else 99
    return 99

final_table["_vord"] = final_table["Variable"].map(var_order).fillna(9).astype(int)
final_table["_cord"] = final_table.apply(cat_rank, axis=1)
final_table = final_table.sort_values(["_vord", "_cord"], kind="stable").drop(columns=["_vord", "_cord"])

out_path = "Table_AWaRe_sensitivity_suspect_only_Poisson_sem_grave.docx"
doc = Document()
style = doc.styles["Normal"]
style.font.name = "Times New Roman"
style._element.rPr.rFonts.set(qn("w:eastAsia"), "Times New Roman")
style.font.size = Pt(11)

title = ("Table S. Sensitivity analysis (suspect antibiotics only): association between AWaRe class "
         "and dose-related medication errors")
p = doc.add_paragraph(title); p.runs[0].bold = True

cols = ["Variable", "Category", "PR (95% CI)", "p-value"]
table = doc.add_table(rows=1, cols=len(cols)); table.style = "Table Grid"
hdr = table.rows[0].cells
for j, c in enumerate(cols):
    hdr[j].text = c; hdr[j].paragraphs[0].runs[0].bold = True

last_var = None
for _, r in final_table.iterrows():
    row = table.add_row().cells
    var = r["Variable"]
    row[0].text = var if var != last_var else ""
    row[1].text = str(r["Category"])
    row[2].text = str(r["PR (95% CI)"])
    row[3].text = str(r["p-value"])
    last_var = var

doc.add_paragraph("")
note = (
    "Note: Sensitivity analysis restricted to drug–event pairs in which the antibiotic was reported "
    "as a suspect medication (RELACAO_MEDICAMENTO_EVENTO = 'Suspeito'). Adjusted prevalence ratios (PR) "
    "and 95% confidence intervals (CI) were estimated using Poisson regression with log link and robust "
    "variance (clustered by notification (IDENTIFICACAO_NOTIFICACAO)). Models were adjusted for age group, "
    "sex, and reporting source. Reference categories: Access (AWaRe class), Adult (19–64) (age), "
    "Male (sex), Healthcare services (reporting source)."
)
note_p = doc.add_paragraph(note); note_p.runs[0].italic = True

doc.save(out_path)
print(f"Saved sensitivity (suspect-only) table: {out_path}")

## Análise temporal

In [ ]:
df_model["ANO_INCLUSAO"] = (df_model["DATA_INCLUSAO_SISTEMA"] // 10000).astype(str)
df_model["ANO_INCLUSAO"].value_counts().sort_index()

In [ ]:
# ============================================================
# ANÁLISE POR ANO (ajuste temporal do Model 2)
# Requer: df_model já criado no script principal (mesma sessão)
# ============================================================
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# --- 0) Checagens de pré-requisito (falha cedo e claro) ---
assert "df_model" in dir(), "df_model não existe: rode o script principal antes."
assert "DATA_INCLUSAO_SISTEMA" in df_model.columns, "Coluna de data ausente em df_model."
assert "ENTRY_EN" in df_model.columns and "SEX_EN" in df_model.columns, \
    "Rode a célula do modelo principal antes (cria ENTRY_EN e SEX_EN)."

ID_COL  = "IDENTIFICACAO_NOTIFICACAO"
AGE_COL = "FAIXA_ETARIA"
Y_COL   = "erro_dose"
SEX_COL = "SEX_EN"

# --- 1) Criar variável de ano (categórica via string) ---
df_model["ANO_INCLUSAO"] = (df_model["DATA_INCLUSAO_SISTEMA"] // 10000).astype(str)

# --- 2) Diagnóstico ANTES de rodar: célula pequena / separação ---
print(">>> Registros por ano x classe:")
print(pd.crosstab(df_model["ANO_INCLUSAO"], df_model["aware_class"]))
print("\n>>> Erro de dose por (ano, classe):")
print(pd.crosstab([df_model["ANO_INCLUSAO"], df_model["aware_class"]], df_model[Y_COL]))

# --- 3) Fórmula base (redefinida aqui p/ não depender da sessão) + ano ---
formula_base = (
    f"{Y_COL} ~ "
    f"C(aware_class, Treatment(reference='ACCESS')) + "
    f"C({AGE_COL}, Treatment(reference='Adulto (19-64 anos)')) + "
    f"C(ENTRY_EN) + "
    f"C(SEX_EN, Treatment(reference='Male'))"
)
formula_ano = formula_base + " + C(ANO_INCLUSAO)"

# --- 4) Rodar modelo ajustado por ano ---
glm_ano = smf.glm(formula=formula_ano, data=df_model, family=sm.families.Poisson())
res_ano = glm_ano.fit(cov_type="cluster", cov_kwds={"groups": df_model[ID_COL]})
print(res_ano.summary())

# --- 5) Extrair PR ---
params_a = res_ano.params
conf_a   = res_ano.conf_int()
pr_table_ano = pd.DataFrame({
    "term": params_a.index,
    "PR": np.exp(params_a.values),
    "CI95_low": np.exp(conf_a[0].values),
    "CI95_high": np.exp(conf_a[1].values),
    "p_value": res_ano.pvalues.values
})
pr_table_ano_no_intercept = pr_table_ano[pr_table_ano["term"] != "Intercept"].copy()
pr_table_ano_no_intercept

In [ ]:
# ==============================================================================
# EXPORTAR TABELA — MODELO TEMPORAL (ajustado por ano de entrada)
# Fonte: pr_table_ano_no_intercept
# ==============================================================================
import re
import pandas as pd
from docx import Document
from docx.shared import Pt
from docx.oxml.ns import qn

tbl = pr_table_ano_no_intercept.copy()

def clean_term(term: str):
    if "aware_class" in term:
        if "RESERVE" in term: return ("AWaRe class", "Reserve")
        if "WATCH" in term:   return ("AWaRe class", "Watch")
    if "FAIXA_ETARIA" in term:
        m = re.search(r"\[T\.(.*)\]", term)
        if m:
            label = m.group(1).strip()
            label = label.replace("Adolescente (13-18 anos)", "Adolescent (13–18)")
            label = label.replace("Criança (6-12 anos)", "Child (6–12)")
            label = label.replace("Infantil (31 dias - 5 anos)", "Infant (31 days–5 years)")
            label = label.replace("Neonato (0-30 dias)", "Neonate (0–30 days)")
            label = label.replace("Idoso (65+ anos)", "Older adult (≥65)")
            label = label.replace("Adulto (19-64 anos)", "Adult (19–64)")
            label = label.replace("Ignorado", "Not reported / Unknown")
            label = label.replace("MISSING", "Not reported / Unknown")
            return ("Age group", label)
    if "ENTRY_EN" in term:
        m = re.search(r"\[T\.(.*)\]", term)
        if m:
            label = m.group(1).strip()
            if label == "Unknown":
                label = "Not reported / Unknown"
            return ("Reporting source", label)
    if "SEX_EN" in term:
        m = re.search(r"\[T\.(.*)\]", term)
        if m:
            return ("Sex", m.group(1).strip())
    if "ANO_INCLUSAO" in term:
        m = re.search(r"\[T\.(.*)\]", term)
        if m:
            return ("Year of entry", m.group(1).strip())
    return ("Other", term)

tbl[["Variable", "Category"]] = tbl["term"].apply(lambda x: pd.Series(clean_term(str(x))))

def fmt_pr_ci(pr, lo, hi, nd=2): return f"{pr:.{nd}f} ({lo:.{nd}f}–{hi:.{nd}f})"
def fmt_p(p): return "" if pd.isna(p) else ("<0.001" if p < 0.001 else f"{p:.3f}")

tbl["PR (95% CI)"] = tbl.apply(lambda r: fmt_pr_ci(r["PR"], r["CI95_low"], r["CI95_high"]), axis=1)
tbl["p-value"] = tbl["p_value"].apply(fmt_p)
tbl_clean = tbl[["Variable", "Category", "PR (95% CI)", "p-value"]].copy()

refs = pd.DataFrame([
    {"Variable": "AWaRe class",      "Category": "Access (reference)",             "PR (95% CI)": "Reference", "p-value": ""},
    {"Variable": "Age group",        "Category": "Adult (19–64; reference)",       "PR (95% CI)": "Reference", "p-value": ""},
    {"Variable": "Sex",              "Category": "Male (reference)",               "PR (95% CI)": "Reference", "p-value": ""},
    {"Variable": "Reporting source", "Category": "Healthcare services (reference)","PR (95% CI)": "Reference", "p-value": ""},
    {"Variable": "Year of entry",    "Category": "2019 (reference)",               "PR (95% CI)": "Reference", "p-value": ""},
])
final_table = pd.concat([refs, tbl_clean], ignore_index=True)

var_order = {"AWaRe class": 0, "Age group": 1, "Sex": 2, "Reporting source": 3, "Year of entry": 4, "Other": 9}
age_order = ["Neonate (0–30 days)", "Infant (31 days–5 years)", "Child (6–12)",
             "Adolescent (13–18)", "Adult (19–64; reference)", "Older adult (≥65)", "Not reported / Unknown"]
sex_order = ["Male (reference)", "Female", "Not reported / Unknown"]
entry_order = ["Healthcare services (reference)", "Healthcare services", "Pharmaceutical companies",
               "Patients and healthcare professionals", "VigiMobile", "VigiFlow eForms",
               "Vaccination services", "Not reported / Unknown"]
aware_order = ["Access (reference)", "Reserve", "Watch"]
year_order = ["2019 (reference)", "2020", "2021", "2022", "2023", "2024", "2025"]

def cat_rank(row):
    v, c = row["Variable"], row["Category"]
    if v == "AWaRe class":      return aware_order.index(c) if c in aware_order else 99
    if v == "Age group":        return age_order.index(c)   if c in age_order   else 99
    if v == "Sex":              return sex_order.index(c)   if c in sex_order   else 99
    if v == "Reporting source": return entry_order.index(c) if c in entry_order else 99
    if v == "Year of entry":    return year_order.index(c)  if c in year_order  else 99
    return 99

final_table["_vord"] = final_table["Variable"].map(var_order).fillna(9).astype(int)
final_table["_cord"] = final_table.apply(cat_rank, axis=1)
final_table = final_table.sort_values(["_vord", "_cord"], kind="stable").drop(columns=["_vord", "_cord"])

out_path = "Table_AWaRe_temporal_adjusted_by_year.docx"
doc = Document()
style = doc.styles["Normal"]
style.font.name = "Times New Roman"
style._element.rPr.rFonts.set(qn("w:eastAsia"), "Times New Roman")
style.font.size = Pt(11)

p = doc.add_paragraph("Table S. Association between AWaRe class and dose-related medication errors, "
                      "additionally adjusted for year of entry into the system")
p.runs[0].bold = True

cols = ["Variable", "Category", "PR (95% CI)", "p-value"]
table = doc.add_table(rows=1, cols=len(cols)); table.style = "Table Grid"
hdr = table.rows[0].cells
for j, c in enumerate(cols):
    hdr[j].text = c; hdr[j].paragraphs[0].runs[0].bold = True

last_var = None
for _, r in final_table.iterrows():
    row = table.add_row().cells
    var = r["Variable"]
    row[0].text = var if var != last_var else ""
    row[1].text = str(r["Category"]); row[2].text = str(r["PR (95% CI)"]); row[3].text = str(r["p-value"])
    last_var = var

doc.add_paragraph("")
note = ("Note: Prevalence ratios (PR) and 95% confidence intervals (CI) from Poisson regression with log link "
        "and robust variance clustered by notification (IDENTIFICACAO_NOTIFICACAO), adjusted for age group, sex, "
        "reporting source, and year of entry. Reference categories: Access (AWaRe class), Adult (19–64) (age), "
        "Male (sex), Healthcare services (reporting source), 2019 (year).")
note_p = doc.add_paragraph(note); note_p.runs[0].italic = True

doc.save(out_path)
print(f"Saved temporal table: {out_path}")

Figura da linha do tempo

In [ ]:
# ============================================================
# FIGURA SUPLEMENTAR: pares por ano (barras) + proporção AWaRe por ano (linhas)
# Requer: df_model com ANO_INCLUSAO (string) já criada
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- 0) Pré-requisitos ---
assert "df_model" in dir(), "df_model não existe: rode o script principal antes."
assert "ANO_INCLUSAO" in df_model.columns, "ANO_INCLUSAO ausente: rode o bloco da análise por ano antes."

AWARE_COL = "aware_class"
aware_order = ["ACCESS", "WATCH", "RESERVE"]
aware_labels = {"ACCESS": "Access", "WATCH": "Watch", "RESERVE": "Reserve"}
aware_colors = {"ACCESS": "#046d4d", "WATCH": "#e67e22", "RESERVE": "#d62728"}
aware_dash   = {"ACCESS": "-", "WATCH": "--", "RESERVE": ":"}   # padrão p/ P&B

# --- 1) Contagem por ano x classe; total por ano; proporção dentro do ano ---
ct = pd.crosstab(df_model["ANO_INCLUSAO"], df_model[AWARE_COL])[aware_order]
total = ct.sum(axis=1)                       # total de pares por ano (barras)
prop  = ct.div(total, axis=0) * 100          # % dentro do ano (linhas)
anos  = ct.index.tolist()

print(prop.round(1))                         # confere os números da figura

# --- 2) Dois painéis empilhados, eixo x compartilhado ---
fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(8, 7), sharex=True,
    gridspec_kw={"height_ratios": [1, 1.2]}
)

# Painel A: barras de total (cinza neutro)
ax_top.bar(anos, total.values, color="#7F77DD", width=0.6)
for x, v in enumerate(total.values):
    ax_top.annotate(f"{int(v)}", (x, v), textcoords="offset points",
                    xytext=(0, 3), ha="center", fontsize=8)
ax_top.set_ylabel("Total drug–event pairs")
ax_top.spines["top"].set_visible(False)
ax_top.spines["right"].set_visible(False)

# Painel B: linhas de proporção (cor + padrão de traço)
for cls in aware_order:
    ax_bot.plot(anos, prop[cls].values, marker="o", markersize=5,
                color=aware_colors[cls], linestyle=aware_dash[cls],
                linewidth=2, label=aware_labels[cls])
ax_bot.set_ylabel("Proportion within year (%)")
ax_bot.set_xlabel("Year of entry into the system")
ax_bot.set_ylim(0, 70)
ax_bot.legend(title="AWaRe class", frameon=False, ncol=3, loc="upper center")
ax_bot.spines["top"].set_visible(False)
ax_bot.spines["right"].set_visible(False)

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.tight_layout()

plt.savefig("Figure_S_pairs_by_year.pdf", format="pdf",
            bbox_inches="tight", facecolor="white")
plt.savefig("Figure_S_pairs_by_year.png", dpi=600,
            bbox_inches="tight", facecolor="white")
plt.show()

## Heatmap

In [ ]:
for k in [2, 3, 5]:
    print(k, int((pivot_total >= k).sum().sum()), "células coloridas")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# ==============================================================================
# 1) PARÂMETROS
# ==============================================================================
MIN_NOTIFICACOES = 5  # mínimo de pares por antibiótico (total) para entrar
MIN_CELL_N = 5   # denominador mínimo do estrato para exibir cor (limiar declarado)

DRUG_COL = "Harmonização"
AWARE_COL = "aware_class"
AGE_COL = "FAIXA_ETARIA"
Y_COL = "erro_dose"

# Ordem AWaRe
aware_order = ["ACCESS", "WATCH", "RESERVE"]

# Mapeamento PT -> EN (ajuste se seus rótulos forem ligeiramente diferentes)
age_map = {
    "Neonato (0-30 dias)": "Neonate (0–30 days)",
    "Infantil (31 dias - 5 anos)": "Infant (31 days–5 years)",
    "Criança (6-12 anos)": "Child (6–12 years)",
    "Adolescente (13-18 anos)": "Adolescent (13–18 years)",
    "Adulto (19-64 anos)": "Adult (19–64 years)",
    "Idoso (65+ anos)": "Older adult (≥65 years)",
    "Ignorado": "Unknown"
}

# Ordem etária desejada (Unknown por último)
age_order_pt = [
    "Neonato (0-30 dias)",
    "Infantil (31 dias - 5 anos)",
    "Criança (6-12 anos)",
    "Adolescente (13-18 anos)",
    "Adulto (19-64 anos)",
    "Idoso (65+ anos)",
    "Ignorado"
]
age_order_en = [age_map[a] for a in age_order_pt]

# ==============================================================================
# 2) PREPARAÇÃO DOS DADOS
# ==============================================================================
# Padronização básica
df_hm = df_model.copy()
df_hm[DRUG_COL] = df_hm[DRUG_COL].astype(str).str.strip()
df_hm[AWARE_COL] = df_hm[AWARE_COL].astype(str).str.strip().str.upper()
df_hm[AGE_COL] = df_hm[AGE_COL].astype(str).str.strip()
df_hm[Y_COL] = df_hm[Y_COL].astype(int)

# Manter apenas AWaRe principal (se desejar; remova se quiser incluir NOT CLASSIFIED/NOT RECOMMENDED)
df_hm = df_hm[df_hm[AWARE_COL].isin(aware_order)].copy()

# Filtrar antibióticos raros (por volume total)
volumetria = df_hm.groupby(DRUG_COL).size()
drugs_to_keep = volumetria[volumetria >= MIN_NOTIFICACOES].index
df_hm = df_hm[df_hm[DRUG_COL].isin(drugs_to_keep)].copy()

# n total por antibiótico (após filtros)
n_by_drug = df_hm.groupby(DRUG_COL).size()

# Traduzir faixa etária para inglês e forçar categoria/ordem
df_hm["AGE_EN"] = df_hm[AGE_COL].map(age_map).fillna(df_hm[AGE_COL])  # fallback se aparecer nível inesperado
df_hm["AGE_EN"] = pd.Categorical(df_hm["AGE_EN"], categories=age_order_en, ordered=True)

# Forçar ordem AWaRe
df_hm[AWARE_COL] = pd.Categorical(df_hm[AWARE_COL], categories=aware_order, ordered=True)

# # ==============================================================================
# # 3) AGREGAR: % erro de dose por (AWaRe, antibiótico, idade)
# # ==============================================================================
# heatmap_data = (
#     df_hm.groupby([AWARE_COL, DRUG_COL, "AGE_EN"], observed=True)[Y_COL]
#          .mean()
#          .mul(100)
#          .reset_index(name="pct_dose")
# )

# # Ordenar linhas: primeiro por AWaRe, depois antibiótico
# heatmap_data = heatmap_data.sort_values([AWARE_COL, DRUG_COL])

# ==============================================================================
# 3) AGREGAR: % erro de dose + n erro de dose por (AWaRe, antibiótico, idade)
# ==============================================================================
agg = (
    df_hm.groupby([AWARE_COL, DRUG_COL, "AGE_EN"], observed=True)[Y_COL]
         .agg(pct_dose="mean", n_dose_err="sum", n_total="size")
         .reset_index()
)

agg["pct_dose"] = agg["pct_dose"] * 100

# Ordenar linhas: primeiro por AWaRe, depois antibiótico
agg = agg.sort_values([AWARE_COL, DRUG_COL])

# ==============================================================================
# 4) PIVOT: linhas = (AWaRe, antibiótico), colunas = idade (EN)
# ==============================================================================
pivot_pct = (
    agg.pivot_table(
        index=[AWARE_COL, DRUG_COL],
        columns="AGE_EN",
        values="pct_dose"
    )
    .reindex(columns=age_order_en)
)

pivot_n = (
    agg.pivot_table(
        index=[AWARE_COL, DRUG_COL],
        columns="AGE_EN",
        values="n_dose_err"
    )
    .reindex(columns=age_order_en)
)

# Para o eixo Y: antibiótico + n total (após filtros)
y_labels = []
for aware_cls, drug in pivot_pct.index:
    n = int(n_by_drug.get(drug, 0))
    y_labels.append(f"{drug} ({n})")

## ==============================================================================
# 5) PLOT
# ==============================================================================
pivot_total = (
    agg.pivot_table(
        index=[AWARE_COL, DRUG_COL],
        columns="AGE_EN",
        values="n_total"
    )
    .reindex(columns=age_order_en)
)

n_drugs = len(pivot_pct)
fig_height = max(10, n_drugs * 0.25)

fig, ax = plt.subplots(figsize=(12, fig_height))

# Estrato inexistente (sem registros) -> mascarado (branco)
mask = pivot_total.isna() | (pivot_total == 0)

# Inteiros seguros
n_int   = pivot_n.fillna(0).round(0).astype(int)       # numerador (erros de dose)
tot_int = pivot_total.fillna(0).round(0).astype(int)   # denominador (total do estrato)
pct_int = pivot_pct.fillna(0).round(0).astype(int)     # %

exists   = ~mask
has_err  = exists & (n_int > 0)
zero_err = exists & (n_int == 0)

# Anotação com DENOMINADOR: "num/den (%)"  ou  "0/den"
annot_text = pd.DataFrame("", index=pivot_pct.index, columns=pivot_pct.columns)
annot_text = annot_text.mask(
    has_err,
    n_int.astype(str) + "/" + tot_int.astype(str) + " (" + pct_int.astype(str) + "%)"
)
annot_text = annot_text.mask(zero_err, "0/" + tot_int.astype(str))

import matplotlib.patches as patches

mask = pivot_total.isna() | (pivot_total == 0)          # estrato NÃO existe
suppressed = (~mask) & (pivot_total < MIN_CELL_N)        # existe mas < limiar

cmap = plt.get_cmap("RdYlGn_r").copy()
cmap.set_bad("white")                                    # base: NaN/inexistente = BRANCO

pivot_color = pivot_pct.where(pivot_total >= MIN_CELL_N) # só colore >= limiar

ax.set_facecolor("white")
sns.heatmap(
    pivot_color, ax=ax, mask=mask, annot=False, cmap=cmap,
    cbar_kws={"label": "% of medication-error records that are dose-related"},
    linewidths=0.5, linecolor="lightgray",
)

# Cinza SÓ nas células suprimidas (existem, mas < limiar)
for i in range(suppressed.shape[0]):
    for j in range(suppressed.shape[1]):
        if bool(suppressed.iat[i, j]):
            ax.add_patch(patches.Rectangle(
                (j, i), 1, 1, facecolor="#f0f0f0",
                edgecolor="lightgray", linewidth=0.5, zorder=3))

# Anotar todas as células existentes (coloridas E cinzas)
for i in range(annot_text.shape[0]):
    for j in range(annot_text.shape[1]):
        txt = annot_text.iat[i, j]
        if txt:
            ax.text(j + 0.5, i + 0.5, txt, ha="center", va="center",
                    fontsize=8, color="black", zorder=4)

# Remover título
ax.set_title("")

# Rótulos dos eixos
ax.set_xlabel("Age group")
ax.set_ylabel(r"Antibiotic ($\it{n}$ = total medication error records)")
ax.yaxis.labelpad = 30

aware_colors = {
    "ACCESS": "#046d4d",   # verde forte
    "WATCH": "#e67e22",    # laranja (melhor que amarelo puro)
    "RESERVE": "#d62728"   # vermelho consistente
}

# Trocar labels do eixo Y para apenas antibiótico
ax.set_yticklabels(y_labels, rotation=0)
for tick_label, (aware_cls, drug) in zip(ax.get_yticklabels(), pivot_pct.index):
    tick_label.set_color(aware_colors.get(aware_cls, "black"))
    tick_label.set_fontweight("bold")
    tick_label.set_fontsize(9)

# Rotacionar x para caber
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")

# ==============================================================================
# 6) Linhas divisórias por AWaRe + rótulo de seção (ajustado para não sobrepor antibióticos)
# ==============================================================================
aware_series = [idx[0] for idx in pivot_pct.index]
boundaries = []
for i in range(1, len(aware_series)):
    if aware_series[i] != aware_series[i-1]:
        boundaries.append(i)

for b in boundaries:
    ax.hlines(b, *ax.get_xlim(), colors="black", linewidth=1.2)

def block_centers(classes):
    centers = {}
    for cls in aware_order:
        idxs = [i for i, c in enumerate(classes) if c == cls]
        if not idxs:
            continue
        s, e = min(idxs), max(idxs)
        centers[cls] = (s + e) / 2
    return centers

centers = block_centers(aware_series)

# >>> AJUSTE INCORPORADO: mais margem à esquerda + texto mais à esquerda <<<
plt.subplots_adjust(left=0.30)  # cria espaço para os rótulos de classe
x_text = -0.35                  # desloca o texto para fora da lista de antibióticos

for cls, y in centers.items():
    ax.text(
        x_text, y + 0.5,
        cls.title(),
        transform=ax.get_yaxis_transform(),
        ha="right",
        va="center",
        fontsize=11,
        fontweight="bold"
    )

# Preserva as fontes como elementos editáveis/vetoriais no PDF
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

# Salvar
plt.tight_layout()

# PDF vetorial — arquivo principal para submissão
plt.savefig(
    "Figure_3_y.pdf",
    format="pdf",
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)

# PNG em alta resolução — arquivo alternativo
plt.savefig(
    "Figure_3_y.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)

plt.show()

print("Gráfico gerado com sucesso!")
print(f"Foram incluídos {n_drugs} antibióticos (após filtro de n>={MIN_NOTIFICACOES}).")


Contagem de co-exposição: Entre notificações de erro de dose:

Média → grau médio de polifarmácia

Mediana → número típico de medicamentos por notificação

Mín–Máx → amplitude observada

Distribuição → permite descrever proporção de monoterapia vs múltiplos medicamentos

Exemplo de frase (modelo técnico):

The median number of medications per notification was X (mean ± SD: Y ± Z), ranging from A to B.

In [ ]:
import pandas as pd
import numpy as np

ID_COL   = "IDENTIFICACAO_NOTIFICACAO"
DRUG_COL = "Harmonização"

# 1) Contar número de medicamentos distintos por notificação
contagem_meds = (
    df.dropna(subset=[ID_COL, DRUG_COL])
      .groupby(ID_COL)[DRUG_COL]
      .nunique()
      .reset_index(name="n_medicamentos")
)

# 2) Distribuição: quantas notificações têm 1, 2, 3, ...
dist_meds = contagem_meds["n_medicamentos"].value_counts().sort_index()

print("Distribuição do número de medicamentos por notificação:")
print(dist_meds)

# 3) Estatísticas descritivas
valores = contagem_meds["n_medicamentos"]

media   = valores.mean()
mediana = valores.median()
desvio  = valores.std()
minimo  = valores.min()
maximo  = valores.max()

print(f"\nMédia: {media:.2f}")
print(f"Mediana: {mediana}")
print(f"Desvio-padrão: {desvio:.2f}")
print(f"Mínimo: {minimo}")
print(f"Máximo: {maximo}")

estatisticas = {
    "Média": round(media, 2),
    "Mediana": mediana,
    "Desvio-padrão": round(desvio, 2),
    "Mínimo": minimo,
    "Máximo": maximo
}

estatisticas

In [ ]:
#Contagem PT unicos por notificação
import pandas as pd
import numpy as np

ID_COL = "IDENTIFICACAO_NOTIFICACAO"
PT_COL = "PT"

# 1) Contar número de PTs distintos por notificação
contagem_pts = (
    df.dropna(subset=[ID_COL, PT_COL])
      .groupby(ID_COL)[PT_COL]
      .nunique()
      .reset_index(name="n_pts")
)

# 2) Distribuição: quantas notificações têm 1, 2, 3, ... PTs
dist_pts = contagem_pts["n_pts"].value_counts().sort_index()

print("Distribuição do número de PTs por notificação:")
print(dist_pts)

# 3) Estatísticas descritivas
valores = contagem_pts["n_pts"]

media   = valores.mean()
mediana = valores.median()
desvio  = valores.std()
minimo  = valores.min()
maximo  = valores.max()

print(f"\nMédia: {media:.2f}")
print(f"Mediana: {mediana}")
print(f"Desvio-padrão: {desvio:.2f}")
print(f"Mínimo: {minimo}")
print(f"Máximo: {maximo}")

estatisticas_pts = {
    "Média": round(media, 2),
    "Mediana": mediana,
    "Desvio-padrão": round(desvio, 2),
    "Mínimo": minimo,
    "Máximo": maximo
}

estatisticas_pts